In [ ]:
import numpy as np
import pandas as pd
import re
import tensorflow as tf
import matplotlib.pyplot as plt

from google.colab import files
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Embedding, LSTM, Dense
from sklearn.model_selection import train_test_split

print("All libraries imported successfully!")
print("TensorFlow version:", tf.__version__)

All libraries imported successfully!
TensorFlow version: 2.21.0


In [ ]:
uploaded = files.upload()

file_name = next(iter(uploaded))

with open(file_name, "r", encoding="utf-8") as f:
    sentences = f.readlines()

sentences = [
    sentence.strip()
    for sentence in sentences
    if sentence.strip()
]

sentences = sentences[:10000]

print("Total sentences:", len(sentences))
print("\nFirst 5 sentences:")

for sentence in sentences[:5]:
    print(sentence)

Saving 1661-0.txt to 1661-0.txt
Total sentences: 9634

First 5 sentences:
﻿
Project Gutenberg's The Adventures of Sherlock Holmes, by Arthur Conan Doyle
This eBook is for the use of anyone anywhere at no cost and with
almost no restrictions whatsoever.  You may copy it, give it away or
re-use it under the terms of the Project Gutenberg License included


In [ ]:
clean_sentences = []

for sentence in sentences:
    sentence = sentence.lower()

    sentence = re.sub(
        r"[^a-z\s']",
        " ",
        sentence
    )

    sentence = re.sub(
        r"\s+",
        " ",
        sentence
    ).strip()

    if sentence:
        clean_sentences.append(sentence)

print("Cleaned sentences:", len(clean_sentences))

for sentence in clean_sentences[:5]:
    print(sentence)

Cleaned sentences: 9632
project gutenberg's the adventures of sherlock holmes by arthur conan doyle
this ebook is for the use of anyone anywhere at no cost and with
almost no restrictions whatsoever you may copy it give it away or
re use it under the terms of the project gutenberg license included
with this ebook or online at www gutenberg net


In [ ]:
train_sentences, test_sentences = train_test_split(
    clean_sentences,
    test_size=0.20,
    random_state=42
)

tokenizer = Tokenizer(
    num_words=10000,
    oov_token="<OOV>"
)

tokenizer.fit_on_texts(train_sentences)

total_words = min(
    len(tokenizer.word_index) + 1,
    10000
)

print("Training sentences:", len(train_sentences))
print("Testing sentences:", len(test_sentences))
print("Vocabulary size:", total_words - 1)

print("\nSample word mappings:")
print(list(tokenizer.word_index.items())[:10])

Training sentences: 7705
Testing sentences: 1927
Vocabulary size: 7299

Sample word mappings:
[('<OOV>', 1), ('the', 2), ('and', 3), ('i', 4), ('of', 5), ('to', 6), ('a', 7), ('in', 8), ('that', 9), ('it', 10)]


In [ ]:
MAX_LEN = 20

def generate_examples(sentence_list):
    inputs = []
    targets = []

    for sentence in sentence_list:
        tokens = tokenizer.texts_to_sequences([sentence])[0]

        for i in range(1, len(tokens)):
            target = tokens[i]

            # Ignore unknown target words
            if target == 1:
                continue

            previous_words = tokens[
                max(0, i - (MAX_LEN - 1)):i
            ]

            inputs.append(previous_words)
            targets.append(target)

    X_data = pad_sequences(
        inputs,
        maxlen=MAX_LEN - 1,
        padding="pre",
        truncating="pre"
    )

    y_data = np.array(targets)

    return X_data, y_data


X_train, y_train = generate_examples(train_sentences)
X_test, y_test = generate_examples(test_sentences)

print("Training input shape:", X_train.shape)
print("Training target shape:", y_train.shape)
print("Testing input shape:", X_test.shape)
print("Testing target shape:", y_test.shape)

Training input shape: (79662, 19)
Training target shape: (79662,)
Testing input shape: (18976, 19)
Testing target shape: (18976,)


In [ ]:
model = Sequential([
    Input(shape=(MAX_LEN - 1,)),

    Embedding(
        input_dim=total_words,
        output_dim=64
    ),

    LSTM(100),

    Dense(
        total_words,
        activation="softmax"
    )
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ (None, 19, 64)         │       467,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 100)            │        66,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 7300)           │       737,300 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,270,500 (4.85 MB)

 Trainable params: 1,270,500 (4.85 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
Embedding(input_dim=total_words, output_dim=64)

<Embedding name=embedding_2, built=False>

In [ ]:
LSTM(100)

<LSTM name=lstm_2, built=False>

In [ ]:
Dense(total_words, activation="softmax")

<Dense name=dense_2, built=False>

In [ ]:
optimizer="adam"
loss="sparse_categorical_crossentropy"
metrics=["accuracy"]

In [ ]:
history = model.fit(
    X_train,
    y_train,
    epochs=10,
    batch_size=64,
    validation_split=0.1,
    verbose=1
)

Epoch 1/10
1121/1121 ━━━━━━━━━━━━━━━━━━━━ 68s 60ms/step - accuracy: 0.0580 - loss: 6.4627 - val_accuracy: 0.0639 - val_loss: 6.3048
Epoch 2/10
1121/1121 ━━━━━━━━━━━━━━━━━━━━ 65s 58ms/step - accuracy: 0.0781 - loss: 5.9597 - val_accuracy: 0.0905 - val_loss: 6.0634
Epoch 3/10
1121/1121 ━━━━━━━━━━━━━━━━━━━━ 82s 58ms/step - accuracy: 0.1111 - loss: 5.6220 - val_accuracy: 0.1138 - val_loss: 5.9183
Epoch 4/10
1121/1121 ━━━━━━━━━━━━━━━━━━━━ 65s 58ms/step - accuracy: 0.1269 - loss: 5.3714 - val_accuracy: 0.1233 - val_loss: 5.8398
Epoch 5/10
1121/1121 ━━━━━━━━━━━━━━━━━━━━ 83s 59ms/step - accuracy: 0.1397 - loss: 5.1740 - val_accuracy: 0.1298 - val_loss: 5.8117
Epoch 6/10
1121/1121 ━━━━━━━━━━━━━━━━━━━━ 65s 58ms/step - accuracy: 0.1524 - loss: 4.9993 - val_accuracy: 0.1348 - val_loss: 5.8112
Epoch 7/10
1121/1121 ━━━━━━━━━━━━━━━━━━━━ 82s 58ms/step - accuracy: 0.1627 - loss: 4.8402 - val_accuracy: 0.1400 - val_loss: 5.8334
Epoch 8/10
1121/1121 ━━━━━━━━━━━━━━━━━━━━ 65s 58ms/step - accuracy: 0.1709 -

In [ ]:
test_loss, top1_accuracy = model.evaluate(
    X_test,
    y_test,
    batch_size=256,
    verbose=1
)

correct_top5 = 0
total_test = len(y_test)

for start in range(0, total_test, 256):
    end = min(start + 256, total_test)

    batch_X = X_test[start:end]
    batch_y = y_test[start:end]

    probabilities = model.predict(
        batch_X,
        verbose=0
    )

    top5_ids = np.argsort(
        probabilities,
        axis=1
    )[:, -5:]

    correct_top5 += np.sum(
        np.any(top5_ids == batch_y[:, None], axis=1)
    )

top5_accuracy = correct_top5 / total_test

print("\n--- MODEL EVALUATION ---")
print(f"Test Loss: {test_loss:.4f}")
print(f"Top-1 Accuracy: {top1_accuracy * 100:.2f}%")
print(f"Top-5 Accuracy: {top5_accuracy * 100:.2f}%")

75/75 ━━━━━━━━━━━━━━━━━━━━ 9s 121ms/step - accuracy: 0.1411 - loss: 5.5330

--- MODEL EVALUATION ---
Test Loss: 5.5330
Top-1 Accuracy: 14.11%
Top-5 Accuracy: 31.38%


In [ ]:
def predict_next_word(phrase):
    phrase = phrase.lower().strip()

    phrase = re.sub(
        r"[^a-z\s']",
        " ",
        phrase
    )

    phrase = re.sub(
        r"\s+",
        " ",
        phrase
    ).strip()

    sequence = tokenizer.texts_to_sequences([phrase])[0]

    if not sequence:
        print("No known words found. Try another phrase.")
        return None

    padded = pad_sequences(
        [sequence],
        maxlen=MAX_LEN - 1,
        padding="pre",
        truncating="pre"
    )

    probabilities = model.predict(
        padded,
        verbose=0
    )[0]

    top5_ids = np.argsort(probabilities)[-5:][::-1]

    predicted_word = tokenizer.index_word.get(
        int(top5_ids[0]),
        "Unknown"
    )

    print("\nInput phrase:", phrase)
    print("Predicted next word:", predicted_word)

    print("\nTop 5 predictions:")

    for word_id in top5_ids:
        word = tokenizer.index_word.get(
            int(word_id),
            "Unknown"
        )

        confidence = probabilities[word_id] * 100

        print(f"{word}: {confidence:.2f}%")

    return predicted_word

In [ ]:
phrase = input("Enter a phrase: ")

predicted_word = predict_next_word(phrase)

Enter a phrase: i love

Input phrase: i love
Predicted next word: that

Top 5 predictions:
that: 13.28%
to: 12.83%
it: 6.23%
the: 4.66%
at: 3.22%


In [ ]:

phrase = input("Enter a phrase: ").strip()
actual_word = input("Enter actual next word: ").lower().strip()

predicted_word = predict_next_word(phrase)

if predicted_word is not None and actual_word:
    print("\nPredicted:", predicted_word)
    print("Actual:", actual_word)

    if predicted_word == actual_word:
        print("Result: CORRECT")
    else:
        print("Result: WRONG")


Enter a phrase: i like
Enter actual next word: you

Input phrase: i like
Predicted next word: the

Top 5 predictions:
the: 10.83%
a: 9.32%
not: 8.87%
it: 7.97%
no: 4.50%

Predicted: the
Actual: you
Result: WRONG


In [ ]:
test_phrases = [
    "i love",
    "how are",
    "thank you",
    "i want to",
    "once upon a"
]

for phrase in test_phrases:
    predict_next_word(phrase)
    print("-" * 40)


Input phrase: i love
Predicted next word: that

Top 5 predictions:
that: 13.28%
to: 12.83%
it: 6.23%
the: 4.66%
at: 3.22%
----------------------------------------

Input phrase: how are
Predicted next word: as

Top 5 predictions:
as: 6.18%
i: 5.47%
to: 5.08%
it: 4.94%
you: 4.67%
----------------------------------------

Input phrase: thank you
Predicted next word: have

Top 5 predictions:
have: 8.14%
will: 5.44%
that: 4.10%
you: 3.72%
can: 3.53%
----------------------------------------

Input phrase: i want to
Predicted next word: be

Top 5 predictions:
be: 7.24%
me: 4.52%
know: 4.00%
say: 3.02%
do: 2.95%
----------------------------------------

Input phrase: once upon a
Predicted next word: very

Top 5 predictions:
very: 6.36%
man: 5.06%
little: 3.38%
few: 2.33%
small: 1.69%
----------------------------------------


In [ ]:

import numpy as np

# Evaluate model
test_loss, test_accuracy = model.evaluate(
    X_test, y_test, verbose=1
)

print(f"Top-1 Accuracy: {test_accuracy * 100:.2f}%")

# Calculate Top-5 Accuracy
correct_top5 = 0
total = len(y_test)
batch_size = 256

for start in range(0, total, batch_size):
    end = min(start + batch_size, total)

    predictions = model.predict(
        X_test[start:end], verbose=0
    )

    actual_words = y_test[start:end]

    top5_words = np.argsort(
        predictions, axis=1
    )[:, -5:]

    for actual, top5 in zip(actual_words, top5_words):
        if actual in top5:
            correct_top5 += 1

top5_accuracy = correct_top5 / total

print(f"Top-5 Accuracy: {top5_accuracy * 100:.2f}%")


593/593 ━━━━━━━━━━━━━━━━━━━━ 9s 15ms/step - accuracy: 0.1411 - loss: 5.5330
Top-1 Accuracy: 14.11%
Top-5 Accuracy: 31.38%


In [ ]:
import pickle

# Save the trained model
model.save("next_word_lstm.keras")

# Save the tokenizer
with open("tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)

print("Model and tokenizer saved successfully!")

Model and tokenizer saved successfully!


In [ ]:
import tensorflow as tf
import pickle

# Load saved model
loaded_model = tf.keras.models.load_model(
    "next_word_lstm.keras"
)

# Load saved tokenizer
with open("tokenizer.pkl", "rb") as f:
    loaded_tokenizer = pickle.load(f)

print("Model loaded successfully!")
print("Tokenizer loaded successfully!")

ValueError: File not found: filepath=next_word_lstm.keras. Please ensure the file is an accessible `.keras` zip file.